In [1]:
"""
Create another dataset from the original dataset which consist of only numerical values.
"""

import pandas as pd

file_path = "GPA_clean.xlsx"
df = pd.read_excel(file_path)

df.dropna().drop_duplicates(inplace=True)
df.drop(['Xếp loại học tập'], axis=1, inplace=True)

df.rename(columns={
    'GPA_1': 'gpa_1', 'GPA_2': 'gpa_2',
    'GPA_3': 'gpa_3', 'GPA_4': 'gpa_4',
    'GPA_5': 'gpa_5', 'GPA_6': 'gpa_6',
}, inplace=True)

output_path = "Data_GPA_clean.xlsx"
df.to_excel(output_path, index=False)

In [2]:
"""
Use the new dataset created to train an XGBoost/RandomForest model for
each grade range ('gpa_1' -> 'gpa_2'; 'gpa_1 + gpa_2' -> 'gpa_3',...).
Each model is evaluated using MSE criteria and saved using joblib.dump().

To better reflect uncertainty we use mapie.regression.CrossConformalRegressor
to generate a range of possible GPA values instead of a single number.
"""

import joblib
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from mapie.regression import CrossConformalRegressor

df = pd.read_excel("Data_GPA_clean.xlsx")

xgb_model_list, xgb_mse_list, xgb_cv_list = [], [], []
rf_model_list, rf_mse_list, rf_cv_list = [], [], []

for train in range(1, 6):
    X = []
    y = []
    for row in df.values.tolist():
        X.append(row[:train])  # Previous GPA
        y.append(row[train])   # Next GPA

    # Split data into train and test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # =====================
    # Train XGBoost model
    # =====================
    xgb_model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        objective='reg:squarederror',
        random_state=42,
    )
    mapie_xgb = CrossConformalRegressor(
        xgb_model,
        confidence_level=0.95,
        cv=5,
    )
    mapie_xgb.fit_conformalize(X_train, y_train)
    xgb_model_list.append(mapie_xgb)

    # Evaluate model
    y_result, y_interval = mapie_xgb.predict_interval(X_test)
    mse = mean_squared_error(y_test, y_result)
    coverage = ((y_test >= y_interval[:, 0]) & (y_test <= y_interval[:, 1])).mean()
    xgb_mse_list.append(mse)
    xgb_cv_list.append(coverage)
    print(f"XGB: GPA_{train + 1} prediction MSE: {mse:.4f}")
    print(f"XGB: GPA_{train + 1} prediction coverage: {coverage:.4f}")

    width = (y_interval[:, 1] - y_interval[:, 0]).mean()
    print(f"XGB: GPA_{train + 1} width: {width:.4f}")

    # =====================
    # Train RandomForest model
    # =====================
    rf_model = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
    mapie_rf = CrossConformalRegressor(
        rf_model,
        confidence_level=0.95,
        cv=5,
    )
    mapie_rf.fit_conformalize(X_train, y_train)
    rf_model_list.append(mapie_rf)

    # Evaluate model
    y_result, y_interval = mapie_rf.predict_interval(X_test)
    mse = mean_squared_error(y_test, y_result)
    coverage = ((y_test >= y_interval[:, 0]) & (y_test <= y_interval[:, 1])).mean()
    rf_mse_list.append(mse)
    rf_cv_list.append(coverage)
    print(f"RF: GPA_{train + 1} prediction MSE: {mse:.4f}")
    print(f"RF: GPA_{train + 1} prediction coverage: {coverage:.4f}")

    width = (y_interval[:, 1] - y_interval[:, 0]).mean()
    print(f"RF: GPA_{train + 1} width: {width:.4f}")

mean = lambda x: sum(x) / len(x)
print(f"\nXGB average MSE: {mean(xgb_mse_list):.4f}")
print(f"XGB average coverage: {mean(xgb_cv_list):.4f}")
print(f"RF average MSE: {mean(rf_mse_list):.4f}")
print(f"RF average coverage: {mean(rf_cv_list):.4f}")

# Save the models using joblib
_ = joblib.dump(xgb_model_list, "xgb_models.joblib")
_ = joblib.dump(rf_model_list, "rf_models.joblib")

XGB: GPA_2 prediction MSE: 0.3485
XGB: GPA_2 prediction coverage: 0.8622
XGB: GPA_2 width: 2.7718
RF: GPA_2 prediction MSE: 0.3486
RF: GPA_2 prediction coverage: 0.8469
RF: GPA_2 width: 2.7500
XGB: GPA_3 prediction MSE: 0.2471
XGB: GPA_3 prediction coverage: 0.7550
XGB: GPA_3 width: 2.0332
RF: GPA_3 prediction MSE: 0.2446
RF: GPA_3 prediction coverage: 0.7542
RF: GPA_3 width: 1.9964
XGB: GPA_4 prediction MSE: 0.2326
XGB: GPA_4 prediction coverage: 0.7016
XGB: GPA_4 width: 2.0092
RF: GPA_4 prediction MSE: 0.2269
RF: GPA_4 prediction coverage: 0.7085
RF: GPA_4 width: 2.0037
XGB: GPA_5 prediction MSE: 0.1731
XGB: GPA_5 prediction coverage: 0.8012
XGB: GPA_5 width: 2.1798
RF: GPA_5 prediction MSE: 0.1816
RF: GPA_5 prediction coverage: 0.7594
RF: GPA_5 width: 2.0161
XGB: GPA_6 prediction MSE: 0.3931
XGB: GPA_6 prediction coverage: 0.8170
XGB: GPA_6 width: 2.6939
RF: GPA_6 prediction MSE: 0.3848
RF: GPA_6 prediction coverage: 0.7688
RF: GPA_6 width: 2.4079

XGB average MSE: 0.2789
XGB averag